# Mushroom Classification — Training Pipeline

8 species, ~474 images. MobileNetV4 ConvSmall + knowledge distillation + quantization.

**Model:** `mobilenetv4_conv_small.e2400_r224_in1k` (3.8M params, 73.8% ImageNet top-1)

**Pipeline:**
1. **Fine-tune** — MobileNetV4 pretrained on ImageNet → mushroom species
2. **Distill** — EfficientNet-B3 teacher → MobileNetV4 student (soft targets)
3. **Quantize** — int8 PTQ via torchao
4. **Export** — TorchScript + ExecuTorch for mobile/embedded

---

In [8]:
import json
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch {torch.__version__}, device={'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")


PyTorch 2.12.0, device=mps


In [9]:
from pathlib import Path

# --- Output dirs ---
data_dir = Path('data/processed/classification_data')
ckpt_dir = Path('checkpoints')
export_dir = Path('exported_models')
for d in (ckpt_dir, export_dir):
    d.mkdir(parents=True, exist_ok=True)

# --- Model ---
num_classes = 8
image_size = 224
ft_arch = 'mobilenetv4_conv_small.e2400_r224_in1k'
teacher_arch = 'efficientnet_b3'

# --- Fine-tune ---
ft_epochs = 50
ft_probe_epochs = 5
ft_lr = 5e-4
ft_wd = 1e-4
ft_batch = 64
ft_warmup = 5
ft_smoothing = 0.1

# --- Distill ---
distill_epochs = 30
distill_lr = 3e-4
distill_wd = 1e-4
distill_batch = 64
distill_T = 6.0
distill_alpha = 0.8
distill_warmup = 3

# --- Runtime ---
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
workers = 8
seed = 42
patience = 10


---
## Define global variables

In [10]:
import os
import pandas as pd

def ensure_classification_data():
    '''Materialize data_dir/{train,val,test}/<raw_species>/<file>.

    Uses SYMLINKS to the raw images referenced in
    data/processed/metadata.csv. Idempotent at the file level: the link
    pass heals broken symlinks and skips valid symlinks and manual real
    files. Total symlink-tree size stays tiny (~280 KB) regardless of
    dataset size.

    Four-state gate per destination:
      valid symlink       -> skip (n_skipped_existing)
      broken symlink      -> unlink + fall through to re-link (n_unlinked_broken)
      real file (manual)  -> skip (n_skipped_manual)
      missing             -> fall through to os.symlink (n_made)

    Per-file try/except OSError around os.symlink so non-symlink-capable
    filesystems (Windows without admin, SELinux-strict) surface a warning
    and continue instead of aborting.

    ImageFolder derives class names from subdirectory names; using
    raw_species (e.g. Amanita_muscaria) matches data/raw/ convention.
    '''
    csv_path = Path('data/processed/metadata.csv')
    if not csv_path.exists():
        print('  metadata.csv not found; run data-preparation.ipynb first.')
        return
    data_dir.mkdir(parents=True, exist_ok=True)
    for split in ('train', 'val', 'test'):
        (data_dir / split).mkdir(parents=True, exist_ok=True)
    metadata = pd.read_csv(csv_path)
    n_made = 0
    n_skipped_missing = 0
    n_skipped_existing = 0
    n_skipped_manual = 0
    n_unlinked_broken = 0
    n_symlink_fail = 0
    for _, row in metadata.iterrows():
        src = Path(row['image_path'])
        if not src.exists():
            n_skipped_missing += 1
            continue
        dst_dir = data_dir / row['split'] / row['raw_species']
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / src.name
        if dst.is_symlink() or dst.exists():
            if dst.is_symlink() and not dst.exists():
                dst.unlink()
                n_unlinked_broken += 1
            elif dst.is_symlink():
                n_skipped_existing += 1
                continue
            else:
                n_skipped_manual += 1
                continue
        try:
            os.symlink(src.resolve(), dst)
            n_made += 1
        except OSError as e:
            n_symlink_fail += 1
            print('  warn: cannot symlink', src, '->', dst, ':', e)
    n_total = sum(1 for p in data_dir.rglob('*') if p.is_file() or p.is_symlink())
    n_classes = sum(1 for p in (data_dir / 'train').iterdir() if p.is_dir())
    print('  Linked', n_made, 'new (recovered', n_unlinked_broken, 'broken,',
          'skipped', n_skipped_missing, 'missing /', n_skipped_existing,
          'valid /', n_skipped_manual, 'manual,', n_symlink_fail,
          'symlink-failed); classification_data has', n_total, 'file(s) across',
          metadata['split'].nunique(), 'splits /', n_classes, 'species.')

ensure_classification_data()


  Linked 0 new (recovered 0 broken, skipped 0 missing / 2839 valid / 0 manual, 0 symlink-failed); classification_data has 2839 file(s) across 3 splits / 8 species.


---
## Data Loaders

- **Train:** `RandomResizedCrop` + `RandAugment(2, 9)` + `CutMix`/`MixUp`
- **Val/Test:** `Resize(256)` + `CenterCrop(224)` — deterministic

In [11]:
def _mixup_collate(batch):
    """Module-level collate for CutMix+MixUp. Picklable for num_workers > 0.

    Note: this function MUST live at module level (NOT inside dataloaders())
    because PyTorch's DataLoader pickles collate_fn to ship to worker
    processes, and a closure capturing a local-scope name (e.g. the
    previous in-place `collate`) fails with
    `AttributeError: Can't get local object 'dataloaders.<locals>.collate'`
    under ForkingPickler.
    """
    imgs, lbls = torch.utils.data.default_collate(batch)
    if imgs.size(0) > 1:
        # torchvision 0.27+: alpha and num_classes are keyword-only.
        imgs, lbls = v2.MixUp(alpha=0.2, num_classes=num_classes)(
            v2.CutMix(alpha=1.0, num_classes=num_classes)(imgs, lbls)
        )
    return imgs, lbls


def dataloaders(batch_size, mixup=False):
    train_tf = v2.Compose([
        v2.ToImage(),  # PIL/ndarray -> tensor (torchvision 0.27+ requires explicit conversion)
        v2.RandomResizedCrop(image_size, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
        v2.RandomHorizontalFlip(p=0.5), v2.RandomRotation(15, fill=128),
        v2.RandAugment(num_ops=2, magnitude=9),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_tf = v2.Compose([
        v2.ToImage(),  # PIL/ndarray -> tensor (torchvision 0.27+ requires explicit conversion)
        v2.Resize(256), v2.CenterCrop(image_size),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    train = DataLoader(datasets.ImageFolder(data_dir / "train", train_tf),
                       batch_size, shuffle=True, num_workers=workers,
                       pin_memory=True, collate_fn=_mixup_collate if mixup else None)
    val = DataLoader(datasets.ImageFolder(data_dir / "val", eval_tf),
                     batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    test = DataLoader(datasets.ImageFolder(data_dir / "test", eval_tf),
                      batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    return train, val, test

train_loader, val_loader, test_loader = dataloaders(ft_batch, mixup=True)
print(f"Train: {len(train_loader.dataset)}  Val: {len(val_loader.dataset)}  Test: {len(test_loader.dataset)}")
print(f"Classes ({len(train_loader.dataset.classes)}): {train_loader.dataset.classes}")

# --- Sanity audit: catches num_classes drift, broken symlinks, transform + collate errors ---
from collections import Counter
assert len(train_loader.dataset.classes) == num_classes, (
    f'cfg.num_classes={num_classes} != #ImageFolder classes={len(train_loader.dataset.classes)}. '
    'Update cfg.num_classes or re-run data-preprocessing-classification.ipynb.'
)
for name, ds in [('train', train_loader.dataset), ('val', val_loader.dataset), ('test', test_loader.dataset)]:
    counts = {c: int(n) for c, n in Counter(ds.targets).items()}
    print(f'  {name:<5s} per-class: ' + ', '.join(f'{c}={counts.get(i, 0)}' for i, c in enumerate(ds.classes)))

# Eval pipeline smoke (no mixup).
xb_e, yb_e = next(iter(val_loader))
print(f'eval-pipeline batch: x {tuple(xb_e.shape)} {xb_e.dtype} range [{float(xb_e.min()):.2f}, {float(xb_e.max()):.2f}]')
print(f'                     y {tuple(yb_e.shape)} {yb_e.dtype} range [{int(yb_e.min())}, {int(yb_e.max())}]')
assert xb_e.dtype == torch.float32 and xb_e.shape == (ft_batch, 3, image_size, image_size), (
    f'eval batch shape/dtype wrong: got {tuple(xb_e.shape)} {xb_e.dtype}, expected ({ft_batch},3,{image_size},{image_size}) float32.'
)
assert int(yb_e.max()) < num_classes, f'eval label {int(yb_e.max())} out of range for num_classes={num_classes}.'

# Train pipeline smoke (mixup=True) -- exercises CutMix+MixUp chain.
xb_t, yb_t = next(iter(train_loader))
print(f'train-pipeline batch (mixup=True): x {tuple(xb_t.shape)} {xb_t.dtype}')
print(f'                                y type={type(yb_t).__name__}  shape={getattr(yb_t, "shape", None)}')
assert xb_t.dtype == torch.float32 and xb_t.shape == (ft_batch, 3, image_size, image_size), (
    f'train batch shape/dtype wrong: got {tuple(xb_t.shape)} {xb_t.dtype}, expected ({ft_batch},3,{image_size},{image_size}) float32.'
)
# MixUp returns a (mixed_one_hot, permuted_labels, lam) tuple; labels are now one-hot [B, num_classes].
# torchvision 0.27: chained CutMix+MixUp returns (mixed_imgs, mixed_one_hot_tensor of shape (B, num_classes)).
# After `xb_t, yb_t = next(iter(train_loader))` the loader's 2-tuple has been unpacked,
# so yb_t is the SHAPE-(B,num_classes) one-hot label tensor (NOT a 3-tuple).
assert isinstance(yb_t, torch.Tensor) and yb_t.shape == (ft_batch, num_classes) and yb_t.dtype == torch.float32, (
    f'expected mixup one-hot label of shape ({ft_batch}, {num_classes}) float32; got type={type(yb_t).__name__} shape={tuple(yb_t.shape)} dtype={yb_t.dtype}.'
)
print('Data loader sanity OK.')

Train: 1987  Val: 426  Test: 426
Classes (8): ['Agaricus_bisporus', 'Amanita_muscaria', 'Boletus_edulis', 'Cantharellus_cibarius', 'Coprinus_comatus', 'Laetiporus_sulphureus', 'Morchella_esculenta', 'Pleurotus_ostreatus']
  train per-class: Agaricus_bisporus=117, Amanita_muscaria=278, Boletus_edulis=279, Cantharellus_cibarius=279, Coprinus_comatus=279, Laetiporus_sulphureus=278, Morchella_esculenta=198, Pleurotus_ostreatus=279
  val   per-class: Agaricus_bisporus=25, Amanita_muscaria=60, Boletus_edulis=60, Cantharellus_cibarius=60, Coprinus_comatus=59, Laetiporus_sulphureus=59, Morchella_esculenta=43, Pleurotus_ostreatus=60
  test  per-class: Agaricus_bisporus=25, Amanita_muscaria=59, Boletus_edulis=60, Cantharellus_cibarius=60, Coprinus_comatus=60, Laetiporus_sulphureus=60, Morchella_esculenta=42, Pleurotus_ostreatus=60
eval-pipeline batch: x (64, 3, 224, 224) torch.float32 range [-2.12, 2.64]
                     y (64,) torch.int64 range [0, 1]


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/phicks/.local/share/uv/python/cpython-3.12.10-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/phicks/.local/share/uv/python/cpython-3.12.10-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute '_mixup_collate' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/phicks/.local/share/uv/python/cpython-3.12.10-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/phicks/.local/share/uv

RuntimeError: DataLoader worker (pid(s) 95429) exited unexpectedly

---
## Models

| Role | Architecture | Params | Pretrained |
|------|-------------|--------|------------|
| Student | MobileNetV4 ConvSmall | 2.5M | ImageNet-1k (73.8%) |
| Teacher | EfficientNet-B3 | 12.2M | ImageNet-1k |

**Why these choices?**
- MobileNetV4 is designed for mobile/edge (UIB blocks, MQA attention)
- B3 teacher gives a real 5× capacity gap (vs B0 which is only 2×)
- With ~330 training images, fine-tuning a pretrained model drastically outperforms scratch

In [ ]:
import timm
import typing
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

def mobilenetv4():
    return timm.create_model(
        model_name=ft_arch,  # mobilenetv4_conv_small.e2400_r224_in1k,
        pretrained=True,
        num_classes=num_classes,
    )


def efficientnetB3():
    # Load the pretrained weight and finetune from that
    m = efficientnet_b3(
        weights=EfficientNet_B3_Weights.IMAGENET1K_V1
    )
    # Replace the final classifier with the mushroom classifier
    m.classifier[1] = nn.Linear(
        in_features=typing.cast(nn.Linear, m.classifier[1]).in_features, # Grab the in_features from the original classifier
        out_features=num_classes,
    )
    return m

def count_params(m):
    return sum(p.numel() for p in m.parameters())

student = mobilenetv4()
teacher = efficientnetB3()

print(f"Student (MobileNetV4):  {count_params(student):,} params")
print(f"Teacher (EfficientNet-B3): {count_params(teacher):,} params")
print(f"Gap: {count_params(teacher) / count_params(student):.1f}×")


model.safetensors:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /Users/phicks/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 89.8MB/s]

Student (MobileNetV4):  2,503,272 params
Teacher (EfficientNet-B3): 10,708,528 params
Gap: 4.3×


---
## Training Utilities

In [ ]:
def accuracy(out, target, k=1):
    _, pred = out.topk(k, 1, True, True)
    return pred.eq(target.view(1, -1).expand_as(pred))[:k].reshape(-1).float().sum(0).item()

def build_scheduler(optimizer, warmup_epochs, total_epochs):
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-6)
    return SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epochs])

def train_one_epoch(model, loader, criterion, optimizer, scheduler, epoch):
    model.train()
    total_loss, total_acc, n = 0, 0, 0
    for images, targets in tqdm(loader, desc=f"Epoch {epoch}"):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        if isinstance(targets, tuple):  # legacy torchvision (<0.27): returns (mixed_one_hot, permuted_labels, lam)
            loss = criterion(outputs, targets[0])
            acc_val = accuracy(outputs, targets[0].argmax(dim=1))
        else:
            # torchvision 0.27+: chain CutMix+MixUp returns a soft-label Tensor of shape (B, num_classes).
            # accuracy() reshapes via `target.view(1, -1)` which assumes a 1-D int target -- argmax first
            # to recover the dominant class index. When mixup is off (no augmentation collate), targets is
            # already a 1-D int tensor and argmax is skipped.
            loss = criterion(outputs, targets)
            acc_val = accuracy(outputs, targets.argmax(dim=1) if targets.ndim == 2 else targets)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * images.size(0)
        total_acc += acc_val
        n += images.size(0)
    return total_loss / n, total_acc / n

@torch.inference_mode()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc, n = 0, 0, 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        total_loss += criterion(outputs, targets).item() * images.size(0)
        total_acc += accuracy(outputs, targets)
        n += images.size(0)
    return total_loss / n, total_acc / n

def save_checkpoint(model, path, **kwargs):
    torch.save({"state_dict": model.state_dict(), **kwargs}, path)
    print(f"  Checkpoint → {path}")

def load_checkpoint(path, model):
    state = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict(state["state_dict"])
    return model, state

---
## Stage 1: Fine-tune MobileNetV4

**What:** Load ImageNet-pretrained weights, replace classifier head, train on mushroom data.

**Key details:**
- `CrossEntropyLoss` with **label smoothing** (0.1) — reduces overfitting on small dataset
- **Two-phase fine-tuning:** probe (head only, 5 epochs) → full fine-tune (backbone + head)
- **RandAugment + CutMix/MixUp** — strong augmentation (MixUp only in full fine-tune phase)
- Early stopping (patience=10) based on val accuracy
- **Not training from scratch** — with ~330 images, fine-tuning a pretrained model is essential

In [ ]:
torch.manual_seed(seed)
model = mobilenetv4().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=ft_smoothing)

# Phase A: Probe — freeze backbone, train classifier head only
for name, param in model.named_parameters():
    if "head" not in name and "classifier" not in name:
        param.requires_grad_(False)
opt_probe = optim.AdamW(model.parameters(), lr=ft_lr, weight_decay=ft_wd)
_train_probe, _val_probe, _ = dataloaders(ft_batch, mixup=False)
best_acc, stall = 0, 0
for epoch in range(1, ft_probe_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, _train_probe, criterion, opt_probe,
                                              build_scheduler(opt_probe, 0, ft_probe_epochs), epoch)
    val_loss, val_acc = evaluate(model, _val_probe, nn.CrossEntropyLoss())
    print(f"  Probe {epoch:2d}  train_acc: {train_acc:.2f}  val_acc: {val_acc:.2f}  val_loss: {val_loss:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc; stall = 0
    else:
        stall += 1
        if stall >= patience: break
print(f"  Probe done — best val acc: {best_acc:.2f}%")

# Phase B: Full fine-tune — unfreeze all, apply MixUp
for param in model.parameters():
    param.requires_grad_(True)
optimizer = optim.AdamW(model.parameters(), lr=ft_lr, weight_decay=ft_wd)
scheduler = build_scheduler(optimizer, ft_warmup, ft_epochs)
_train_ft, _val_ft, _ = dataloaders(ft_batch, mixup=True)
best_acc, stall = 0, 0
ckpt_path = ckpt_dir / "mobilenetv4_finetuned.pth"
for epoch in range(1, ft_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, _train_ft, criterion, optimizer, scheduler, epoch)
    val_loss, val_acc = evaluate(model, _val_ft, nn.CrossEntropyLoss())
    print(f"  Epoch {epoch:2d}  train_acc: {train_acc:.2f}  val_acc: {val_acc:.2f}  val_loss: {val_loss:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        save_checkpoint(model, ckpt_path, epoch=epoch, val_acc=val_acc)
        stall = 0
    else:
        stall += 1
        if stall >= patience:
            print(f"  Early stopping at epoch {epoch}")
            break
print(f"\n✅ Stage 1 complete — best val acc: {best_acc:.2f}%")


---
## Stage 2: Knowledge Distillation

**What:** Teach MobileNetV4 to match EfficientNet-B3's softmax distributions.

**Why distill after fine-tuning?**
- Fine-tuning already gives a strong baseline
- Distillation adds dark knowledge (inter-class relationships: "this looks like _Boletus edulis_ but also a bit like _Agaricus bisporus_")
- With T=6.0, the student sees richer probability distributions from the teacher
- α=0.8 means 80% of the loss comes from matching the teacher, 20% from ground-truth

**Teacher gap:** EfficientNet-B3 has 5× more parameters than MobileNetV4 — large enough for meaningful transfer.

In [ ]:
teacher_model = efficientnet_b3().to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad_(False)

student_model = mobilenetv4().to(device)
ft_path = ckpt_dir / "mobilenetv4_finetuned.pth"
if ft_path.exists():
    load_checkpoint(ft_path, student_model)
    print(f"Loaded fine-tuned weights from {ft_path}")

def distillation_loss(s_logits, t_logits, targets):
    ce = nn.CrossEntropyLoss(label_smoothing=ft_smoothing)(s_logits, targets)
    kl = F.kl_div(
        F.log_softmax(s_logits / distill_T, dim=1),
        F.softmax(t_logits / distill_T, dim=1),
        reduction="batchmean",
    ) * (distill_T ** 2)
    return distill_alpha * kl + (1 - distill_alpha) * ce

opt = optim.AdamW(student_model.parameters(), lr=distill_lr, weight_decay=distill_wd)
sched = build_scheduler(opt, distill_warmup, distill_epochs)
_train_dist, _val_dist, _ = dataloaders(distill_batch, mixup=True)

best_acc, stall = 0, 0
ckpt_path = ckpt_dir / "mobilenetv4_distilled.pth"

for epoch in range(1, distill_epochs + 1):
    student_model.train()
    total_loss, total_acc, n = 0, 0, 0
    for images, targets in tqdm(train_loader, desc=f"Distill Epoch {epoch}"):
        images, targets = images.to(device), targets.to(device)
        with torch.no_grad():
            t_logits = teacher_model(images)
        opt.zero_grad()
        s_logits = student_model(images)
        if isinstance(targets, tuple):  # legacy torchvision (<0.27): returns (mixed_one_hot, permuted_labels, lam)
            loss = distillation_loss(s_logits, t_logits, targets[0])
            acc_val = accuracy(s_logits, targets[0].argmax(dim=1))
        else:
            # torchvision 0.27+: train_loader targets is a shape-(B, num_classes) soft-label Tensor
            # after the CutMix+MixUp chain. argmax to recover the dominant class index for accuracy.
            loss = distillation_loss(s_logits, t_logits, targets)
            acc_val = accuracy(s_logits, targets.argmax(dim=1) if targets.ndim == 2 else targets)
        loss.backward(); opt.step(); sched.step()
        total_loss += loss.item() * images.size(0)
        total_acc += acc_val
        n += images.size(0)
    
    val_loss, val_acc = evaluate(student_model, _val_dist, nn.CrossEntropyLoss())
    print(f"  Epoch {epoch:2d}  train_acc: {total_acc/n:.2f}  val_acc: {val_acc:.2f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        save_checkpoint(student_model, ckpt_path, epoch=epoch, val_acc=val_acc)
        stall = 0
    else:
        stall += 1
        if stall >= patience:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\n✅ Stage 2 complete — best val acc: {best_acc:.2f}%")

---
## Stage 3: Post-Training Quantization

**What:** Convert the distilled model to int8 using `torchao`.

**Why:** int8 models are ~4× smaller and 2-4× faster on CPU/mobile. For a ~330-image dataset, PTQ with `int8_weight_only` is sufficient — no need for QAT.

> Requires: `pip install torchao`

In [ ]:
model = mobilenetv4().to("cpu").eval()

src = ckpt_dir / "mobilenetv4_distilled.pth"
if not src.exists():
    src = ckpt_dir / "mobilenetv4_finetuned.pth"
assert src.exists(), f"No checkpoint found at {src}"
model.load_state_dict(torch.load(src, map_location="cpu", weights_only=True)["state_dict"])
print(f"Loaded from {src}")

try:
    from torchao.quantization import quantize_, int8_weight_only
    quantize_(model, int8_weight_only())
    save_checkpoint(model, ckpt_dir / "mobilenetv4_quantized.pth")
    print("✅ Quantization complete (int8 weight-only)")
except ImportError:
    print("❌ torchao not installed. Run: pip install torchao")

---
## Stage 4: Export for Mobile/Embedded

**What:** Export to two formats:
- **TorchScript (.pt)** — general-purpose, works with C++ LibTorch
- **ExportedProgram (.ep)** — for [ExecuTorch](https://pytorch.org/executorch) deployment on mobile/embedded

**Why ExecuTorch over TFLite?** ExecuTorch is PyTorch-native (no format conversion), GA since 2025, supports 12+ hardware backends (CoreML, QNN, XNNPACK, Metal).

In [ ]:
model = mobilenetv4().to("cpu").eval()

for src_name in ["mobilenetv4_quantized.pth", "mobilenetv4_distilled.pth", "mobilenetv4_finetuned.pth"]:
    src = ckpt_dir / src_name
    if src.exists():
        model.load_state_dict(torch.load(src, map_location="cpu", weights_only=True)["state_dict"])
        print(f"Loaded from {src}")
        break
else:
    raise FileNotFoundError("No checkpoint found")

dummy = torch.randn(1, 3, image_size, image_size)

# TorchScript
traced = torch.jit.trace(model, dummy)
traced.save(str(export_dir / "mobilenetv4.pt"))
print(f"✅ TorchScript → {export_dir / 'mobilenetv4.pt'}")

# ExecuTorch
try:
    ep = torch.export.export(model, (dummy,))
    ep.save(str(export_dir / "mobilenetv4.ep"))
    print(f"✅ ExecuTorch  → {export_dir / 'mobilenetv4.ep'}")
except Exception as e:
    print(f"⚠️  ExecuTorch export skipped: {e}")

---
## Summary & Next Steps

| Stage | Output | Status |
|-------|--------|--------|
| 1. Fine-tune | `checkpoints/mobilenetv4_finetuned.pth` | |
| 2. Distill | `checkpoints/mobilenetv4_distilled.pth` | |
| 3. Quantize | `checkpoints/mobilenetv4_quantized.pth` | |
| 4. Export | `exported_models/mobilenetv4.{pt,ep}` | |

**To push to cloud GPU:**
```bash
rsync -avz . user@cloud-vm:~/mushroom-project/
ssh user@cloud-vm
cd ~/mushroom-project
python train_classification.py
```

**To use ExecuTorch on device:**
```bash
pip install executorch
python -m executorch.examples.export --model_path exported_models/mobilenetv4.ep
```

**Reference:** MobileNetV4 paper: [arXiv 2404.10518](https://arxiv.org/abs/2404.10518)

In [ ]:
summary = {
    "model": ft_arch,
    "teacher": teacher_arch,
    "num_classes": num_classes,
    "image_size": image_size,
    "checkpoint_dir": str(ckpt_dir),
    "export_dir": str(export_dir),
    "stages": ["finetune", "distill", "quantize", "export"],
}
with open(ckpt_dir / "pipeline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))